toca sdacatrle el promedio a cada coeficiente y su varianza cada una en un vector y luego concatenar esos vectores y hacer una matriz por el numAudios x 72(36p y 36 DE)

otra matriz con la caracterizacion de cada audio x 19k





In [20]:
from pathlib import Path
import librosa
import matplotlib.pyplot as plt
import librosa.display

carpeta_audios = Path("Dataset") / "Audios" / "Guitarra"

archivos = list(carpeta_audios.glob("*.wav"))

for archivo in archivos[:100]:
    arreglo, frec_muestreo = librosa.load(archivo, sr=None)
    print("Procesando:", archivo.name)


Procesando: 1.wav
Procesando: 10.wav
Procesando: 100.wav
Procesando: 1000.wav
Procesando: 1001.wav
Procesando: 1002.wav
Procesando: 1003.wav
Procesando: 1004.wav
Procesando: 1005.wav
Procesando: 1006.wav
Procesando: 1007.wav
Procesando: 1008.wav
Procesando: 1009.wav
Procesando: 101.wav
Procesando: 1010.wav
Procesando: 1011.wav
Procesando: 1012.wav
Procesando: 1013.wav
Procesando: 1014.wav
Procesando: 1015.wav
Procesando: 1016.wav
Procesando: 1017.wav
Procesando: 1018.wav
Procesando: 1019.wav
Procesando: 102.wav
Procesando: 1020.wav
Procesando: 1021.wav
Procesando: 1022.wav
Procesando: 1023.wav
Procesando: 1024.wav
Procesando: 1025.wav
Procesando: 1026.wav
Procesando: 1027.wav
Procesando: 1028.wav
Procesando: 1029.wav
Procesando: 103.wav
Procesando: 1030.wav
Procesando: 1031.wav
Procesando: 1032.wav
Procesando: 1033.wav
Procesando: 1034.wav
Procesando: 1035.wav
Procesando: 1036.wav
Procesando: 1037.wav
Procesando: 1038.wav
Procesando: 1039.wav
Procesando: 104.wav
Procesando: 1040.wav
Pr

Preenfasis


In [12]:
import numpy as np
y = arreglo
preenfasis = 0.97
y_preenfatizada = np.append(y[0],y[1:]-preenfasis*y[:-1])

Ventaneo o framming con windowing

In [13]:
tamano_cuadro = 0.025 #25ms
paso_cuadro= 0.01 #10ms    --> para solapar(suavizar)

sr = frec_muestreo
#convertir de segundos a muestras
longitud_cuadro, paso_cuadros_muestras = tamano_cuadro*sr, paso_cuadro*sr
longitud_senal = len(y_preenfatizada)

#redondear enteros
longitud_cuadro = int(longitud_cuadro)
paso_cuadros_muestras = int(paso_cuadros_muestras)

#calcular mel numero de cuadros
num_cuadros = int(np.ceil(float(np.abs(longitud_senal-longitud_cuadro))/paso_cuadros_muestras))

#rellenar para que se asegure que todos los frames tengan el mismo numero de muestras
long_senal_rellenar = num_cuadros*paso_cuadros_muestras+longitud_cuadro
ceros_rellenar = np.zeros(long_senal_rellenar-longitud_senal)
senal_rellena = np.append(y_preenfatizada,ceros_rellenar)

#Dividir la señal en frames
indices=  np.tile(np.arange(0,longitud_cuadro),(num_cuadros,1)) + \
          np.tile(np.arange(0, num_cuadros*paso_cuadros_muestras, paso_cuadros_muestras),(longitud_cuadro,1)).T
cuadros = senal_rellena[indices.astype(np.int32, copy=False)]

print(indices)
print(len(cuadros))
cuadros *= np.hamming(longitud_cuadro)


[[    0     1     2 ...   548   549   550]
 [  220   221   222 ...   768   769   770]
 [  440   441   442 ...   988   989   990]
 ...
 [65120 65121 65122 ... 65668 65669 65670]
 [65340 65341 65342 ... 65888 65889 65890]
 [65560 65561 65562 ... 66108 66109 66110]]
299


APLICAR LA VENTANA DE HAMMING

CLASE 17 FEBRERO CONTINUACION

In [14]:
#Transformada de fourier
NFFT=512 #numerto de puntos de la transformada de fourier

#calcular magnitud y el espectro de potencia
magnitud_cuadros = np.abs(np.fft.fft(cuadros,NFFT)) #magnitud
potencia_cuadros = ((1.0/NFFT)*((magnitud_cuadros)**2)) #espectro potencia

print (magnitud_cuadros.shape)

(299, 512)


ESPECTOGRAMA

In [15]:
#APLICACION DEL BANCO DE FILTROS MEL

#definir numero de filtros
num_filtros = 40

#limites frecuencia en escala de Mel
mel_frec_baja=0
mel_frec_alta=2595*np.log10(1+(sr/2)/700) #conversion de Hz a mEL

#Calcular puntos igualmente espaciados en la escala de Mel
puntos_mel = np.linspace(mel_frec_baja, mel_frec_alta, num_filtros+2)

#convertir Mel a Haz
puntos_hz = (700*(10**(puntos_mel/2595)-1))

#calcular los bins de frecuencia para la FFT
bins = np.floor((NFFT+1)*puntos_hz/sr)
bins = bins.astype(int)


#inicializar la matriz del banco de filtros
banco_filtros = np.zeros((num_filtros, int(np.floor(NFFT))))

#construir los filtros triangulares Mel
for m in range(1, num_filtros+1):
  f_m_menos = int(bins[m-1]) #limite izquierdo
  f_m = int(bins[m]) #centro
  f_m_mas = int(bins[m+1]) #limite derecho

  #pendiente4 ascendente
  for k in range(f_m_menos, f_m):
    banco_filtros[m-1,k] = (k-bins[m-1])/(bins[m]-bins[m-1])

  #pendiente4 descendente
  for k in range(f_m, f_m_mas):
    banco_filtros[m-1,k] = (bins[m+1]-k)/(bins[m+1]-bins[m])

#Aplicar los filtros a la potencia de los frames
energia_filtros = np.dot(potencia_cuadros, banco_filtros.T)

#para evitar log de cero 0 --> coloca valores muy pequeños cercanos a 0 pero no es 0
energia_filtros = np.where(energia_filtros == 0, np.finfo(float).eps, energia_filtros)

#convertir a escala logaritmica
energia_filtros = 20*np.log10(energia_filtros)

TRANSFORMADA DISCRETA DEL COSENO (DCT)


In [16]:
#Calcular transformada diuscreta del coseno

from scipy.fftpack import dct

#numero de coeficientes cepstrales a calcular
num_ceps = 12

mfcc = dct(energia_filtros, type=2, axis=1, norm='ortho')[:, :num_ceps]

In [17]:
#Deltas (primera derivada)
delta_mfcc = librosa.feature.delta(mfcc, order=1, width=9)
#Deltas-deltas(segunda derivada)
delta2_mfcc = librosa.feature.delta(mfcc, order=2, width=9)

In [18]:
#Vector de caracteristicas

caracteristicas = np.concatenate((mfcc, delta_mfcc, delta2_mfcc), axis=1)
print(caracteristicas.shape)

(299, 36)


In [19]:
#Normalizar cepstrales  (CMVN)
carac_norm = (caracteristicas - np.mean(caracteristicas, axis=0))/np.std(caracteristicas, axis=0)
print(caracteristicas[0])
print(carac_norm [0])

[-4.39503777e+02  2.46052088e+01 -8.25443651e+01  5.46862731e+01
 -4.64445433e+00 -2.10388440e+01 -2.90323796e+01 -1.52323944e+01
  9.77353178e+00 -2.47929460e+01 -3.29663481e+01 -1.74456608e+01
  2.84815880e+01  2.84815880e+01  2.84815880e+01  2.84815880e+01
  2.84815880e+01 -1.41440314e+00 -9.13861280e-02 -5.70326183e+00
 -5.70326183e+00 -5.70326183e+00 -5.70326183e+00 -5.70326183e+00
 -2.50072303e+01 -2.50072303e+01 -2.50072303e+01 -2.50072303e+01
 -2.50072303e+01  3.52812882e-01 -4.04506635e+00  3.84884015e+00
  3.84884015e+00  3.84884015e+00  3.84884015e+00  3.84884015e+00]
[ 0.56472008 -1.26933218  1.25956992  0.46766789  0.94331351  1.07415609
 -0.08071888  0.72782051  2.09102531 -0.50564137 -0.83558625 -0.4114274
  0.05771606  0.05771606  0.05771606  0.05771606  0.05771606  0.88975953
 -1.19020536 -1.32929825 -1.32929825 -1.32929825 -1.32929825 -1.32929825
  0.41403095  0.41403095  0.41403095  0.41403095  0.41403095 -1.80761304
 -0.6294985  -0.81817629 -0.81817629 -0.81817629 -